This is a workflow to find starting points for ATS 2D transect with installed site lat&lon

Determine lat and lon of the starting and ending points of 2D transect

Input information
- lat and lon of the installed sites from field team
- hydrography; NHD plus; HUC and river networks
- DEM
- `config.json`

Output
- `./data/dem/reprojected_dem.tif`
- m2_mat_filename = `../data-processed/{site_name}/startendcoords_{site_name}.mat`
    - `start_coords` and `end_coords`
- it's a perpendicular line representing a hillslope which ends at a specified installed site


In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import warnings
warnings.filterwarnings('ignore', module='pyproj')

# Parameters and data sources

In [ ]:
# Parameters cell
import json
with open('config.json', 'r') as f:
    config = json.load(f)
watershed_name = config['watershed_name']
hucs           = [config['hucs']]
site_name      = config['site_name']

meshsize_nx = config['meshsize_nx']

In [ ]:
def get_huc12(hucs):
    huc12_list = []
    for huc in hucs:
        if len(huc) == 12:
            huc12_list.append(huc)
        elif len(huc) == 10:
            for i in range(1,20):
                huc12_list.append(huc+str(i).zfill(2))
        elif len(huc) == 8:
            for i in range(1,20):
                for j in range(1,20):
                    huc12_list.append(huc+str(i).zfill(2)+str(j).zfill(2))
        else:
            print('need huc8, huc10 or huc12')
    return huc12_list

hucs = get_huc12(hucs)
print(hucs[:10])

In [ ]:
# Parameters cell -- this provides all parameters that can be changed via pipelining to generate a new watershed.
huc_level = 12 # if provided, an int setting the level at which to include HUC boundaries

# geometric parameters
# simplify_hucs = 80 # length scale to target average edge
# simplify_rivers = 30
# stream_outlet_width = 500 # half-width to track a labeled set on which to get discharge
ignore_small_rivers = 2 #default=2 # ignore rivers which have this or fewer reaches.  likely they are irrigation ditches
                        # or other small features which make things complicated but likely don't add much value
prune_by_area_fraction = 0.0 #default=0.01 # ignore reaches whose accumulated catchment area is less than this fraction of the
                              # full domain's area
prune_by_area_fraction_waterbodies = None
# num_smoothing_sweeps = 2 # number of times to smooth the DEM prior to elevating

# # triangle refinement control
include_rivers = True
# # refine_d0 = 100
# # refine_d1 = 500
# # refine_A0 = 8000
# # refine_A1 = 50000
# meshsize = 100
# factor = 5
# refine_d0 = meshsize*3
# refine_A0 = meshsize**2/2
# refine_d1 = meshsize*15
# refine_A1 = (np.round(meshsize*factor))**2/2

# logistics
generate_plots = True # plots take time to make and aren't always needed

In [ ]:
# conda package imports
import os,sys
import numpy as np
import shapely
import pyproj
import pandas as pd
import itertools
import rasterio
import rasterio.transform
import rasterio.features
from matplotlib import pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from matplotlib.tri import Triangulation

# Watershed Workflow
import watershed_workflow
import watershed_workflow.source_list
import watershed_workflow.ui
import watershed_workflow.colors
import watershed_workflow.mesh
import watershed_workflow.split_hucs
import watershed_workflow.utils

In [ ]:
# Note that, by default, we tend to work in the DayMet CRS because this allows us to avoid
# reprojecting meteorological forcing datasets.
crs = watershed_workflow.crs.daymet_crs()
crs

In [ ]:
# set up a dictionary of source objects
sources = watershed_workflow.source_list.get_default_sources()
sources['hydrography'] = watershed_workflow.source_list.hydrography_sources['NHD Plus']
sources['HUC'] = watershed_workflow.source_list.huc_sources['NHD Plus']
sources['DEM'] = watershed_workflow.source_list.dem_sources['NED 1/3 arc-second']
#sources['geologic structure'] = watershed_workflow.source_list.FileManagerGLHYMPS('/global/cfs/cdirs/m1800/zhi/ww/scripts/data/soil_structure/GLHYMPS/GLHYMPS.shp')
#sources['depth to bedrock'] = watershed_workflow.source_list.FileManagerRaster('/global/cfs/cdirs/m1800/zhi/ww/scripts/data/soil_structure/SoilGrids2017/BDTICM_M_250m_ll.tif')
#sources['geologic structure'] = watershed_workflow.source_list.FileManagerGLHYMPS('/global/cfs/cdirs/m1800/xiaoyi/ARW-ELMPF/data/ww_data_from_zhi/soil_structure/GLHYMPS/GLHYMPS.shp')
#sources['depth to bedrock'] = watershed_workflow.source_list.FileManagerRaster('/global/cfs/cdirs/m1800/xiaoyi/ARW-ELMPF/data/ww_data_from_zhi/soil_structure/SoilGrids2017/BDTICM_M_250m_ll.tif')
watershed_workflow.source_list.log_sources(sources)

In [ ]:
# Prepare '../data-processed' folders
os.makedirs(f'../data-processed/{watershed_name}', exist_ok=True)
os.makedirs(f'../data-processed/{site_name}', exist_ok=True)

# Prepare './images' folders
os.makedirs(f'./images/{site_name}', exist_ok=True)
os.makedirs(f'./images/{watershed_name}', exist_ok=True)

# Load watershed DEM

In [ ]:
# load the huc
my_hucs = []
for huc in hucs:
    _, ws = watershed_workflow.get_hucs(sources['HUC'], huc, huc_level, crs)
    my_hucs.extend(ws)

watershed = watershed_workflow.split_hucs.SplitHUCs(my_hucs)

In [ ]:
if include_rivers:  
    # download/collect the river network within that shape's bounds
    _, reaches = watershed_workflow.get_reaches(sources['hydrography'], huc, 
                                                watershed.exterior(), crs, crs,
                                                in_network=True, properties=True)
    
    rivers = watershed_workflow.construct_rivers(reaches, method='hydroseq',
                                                 ignore_small_rivers=ignore_small_rivers,
                                                 prune_by_area=prune_by_area_fraction * watershed.exterior().area * 1.e-6,
                                                 remove_diversions=True,
                                                 remove_braided_divergences=True)
else:
    reaches = []
    rivers = []

In [ ]:
# identify outlets by the elevation map
#watershed_workflow.split_hucs.find_outlets_by_elevation(watershed, crs, dem_sm, dem_profile)

print(len(rivers))
rivers = sorted(rivers, key=len)
#watershed_workflow.split_hucs.find_outlets_by_hydroseq(watershed, rivers[-1])

if generate_plots:
    fig, ax = watershed_workflow.plot.get_ax(crs, figsize=(12,10))
    colors = watershed_workflow.colors.enumerated_colors(len(watershed), palette=4)
    #watershed_workflow.plot.hucs(watershed, crs, ax=ax, color=colors, linewidth=0, facecolor='color', alpha=0.4)
    watershed_workflow.plot.hucs(watershed, crs, ax=ax, color=colors, linewidth=0, facecolor='color', alpha=0.4)
    watershed_workflow.plot.rivers(rivers, crs, ax=ax, colors='b', linewidth=0.5)
    #watershed_workflow.plot.shplys(watershed.polygon_outlets, crs, ax=ax, color=colors, marker='o', markersize=200)

pyproj lat/lon of observation sites to x/y
- load site locations from the site availability summary CSV
- preserve dataset-specific coordinates from lin, delgado, and barnes
- compute dataset-specific DayMet x/y for each valid lat/lon pair
- derive plot_lon, plot_lat, plot_x, and plot_y using the first valid source in the order lin -> delgado -> barnes

In [ ]:
# Load site availability summary with dataset-specific coordinates
csv_path = "/global/homes/x/xiao284/cfs_m1800/xiaoyi/10-Projects/2025-RCSFA-HillslopeFire/Pipeline_dev/validate_Naches_ATS_2D_cleaned.obs_sites/notebooks/site_obs/site_availability_summary.csv"

df_points = pd.read_csv(csv_path)

coordinate_columns = [
    'lin_lat', 'lin_lon',
    'delgado_lat', 'delgado_lon',
    'barnes_lat', 'barnes_lon',
]
df_points[coordinate_columns] = df_points[coordinate_columns].apply(pd.to_numeric, errors='coerce')

# Set up coordinate transformation
wgs84 = pyproj.Proj(proj='latlong', datum='WGS84')
proj_daymet = pyproj.Proj('+proj=lcc +lat_1=25 +lat_2=60 +lat_0=42.5 +lon_0=-100 +x_0=0 +y_0=0 +ellps=WGS84 +units=m +no_defs')

source_columns = {
    'lin': ('lin_lon', 'lin_lat'),
    'delgado': ('delgado_lon', 'delgado_lat'),
    'barnes': ('barnes_lon', 'barnes_lat'),
}

def project_to_daymet(lon, lat):
    if pd.isna(lon) or pd.isna(lat):
        return np.nan, np.nan
    return pyproj.transform(wgs84, proj_daymet, lon, lat)

# Compute dataset-specific projected coordinates first
for source_name, (lon_col, lat_col) in source_columns.items():
    x_coords = []
    y_coords = []
    for lon, lat in zip(df_points[lon_col], df_points[lat_col]):
        x, y = project_to_daymet(lon, lat)
        x_coords.append(x)
        y_coords.append(y)
    df_points[f'{source_name}_x'] = x_coords
    df_points[f'{source_name}_y'] = y_coords

# Derive plotting coordinates using first valid source: lin -> delgado -> barnes
plot_lons = []
plot_lats = []
plot_xs = []
plot_ys = []
coord_sources = []
missing_sites = []

for _, row in df_points.iterrows():
    selected_source = None
    selected_lon = np.nan
    selected_lat = np.nan
    selected_x = np.nan
    selected_y = np.nan

    for source_name, (lon_col, lat_col) in source_columns.items():
        lon = row[lon_col]
        lat = row[lat_col]
        if pd.notna(lon) and pd.notna(lat):
            selected_source = source_name
            selected_lon = lon
            selected_lat = lat
            selected_x = row[f'{source_name}_x']
            selected_y = row[f'{source_name}_y']
            break

    if selected_source is None:
        missing_sites.append(row['site_id'])

    coord_sources.append(selected_source)
    plot_lons.append(selected_lon)
    plot_lats.append(selected_lat)
    plot_xs.append(selected_x)
    plot_ys.append(selected_y)

if missing_sites:
    raise ValueError(
        'No valid lat/lon pair found for site_id values: ' + ', '.join(missing_sites)
    )

df_points['coord_source'] = coord_sources
df_points['plot_lon'] = plot_lons
df_points['plot_lat'] = plot_lats
df_points['plot_x'] = plot_xs
df_points['plot_y'] = plot_ys

print("\n" + "=" * 90)
print("OBSERVATION SITE COORDINATES (DayMet CRS)")
print("=" * 90)
print(f"\n{'Site':<12} {'Source':<10} {'Plot Lon':<12} {'Plot Lat':<12} {'Plot X [m]':<15} {'Plot Y [m]':<15}")
print(f"{'-' * 12} {'-' * 10} {'-' * 12} {'-' * 12} {'-' * 15} {'-' * 15}")

for _, row in df_points.iterrows():
    print(
        f"{row['site_id']:<12} {row['coord_source']:<10} "
        f"{row['plot_lon']:<12.6f} {row['plot_lat']:<12.6f} "
        f"{row['plot_x']:<15.2f} {row['plot_y']:<15.2f}"
    )

In [ ]:
# Plot all observation sites using the first valid source-specific coordinates
source_colors = {
    'lin': 'tab:blue',
    'delgado': 'tab:orange',
    'barnes': 'tab:green',
}

sites = {}
for _, row in df_points.iterrows():
    sites[row['site_id']] = {
        'name': row['site_id'],
        'coord_source': row['coord_source'],
        'x': row['plot_x'],
        'y': row['plot_y'],
    }

fig, ax = watershed_workflow.plot.get_ax(crs, figsize=(9,7.5))
watershed_workflow.plot.hucs(watershed, crs, ax=ax, color=colors[0], linewidth=1, facecolor='color', alpha=0.4)
watershed_workflow.plot.rivers(rivers, crs, ax=ax, colors='b', linewidth=0.5)

legend_handles = {}

for site_id, site in sites.items():
    coord_source = site['coord_source']
    color = source_colors.get(coord_source, 'black')

    handle = ax.plot(
        site['x'], site['y'], 'o',
        color=color,
        markersize=8, markeredgecolor='none', markeredgewidth=1,
    )[0]
    ax.text(site['x'], site['y'] - 1000, site_id, ha='left', va='top', fontsize=8)

    if coord_source not in legend_handles:
        legend_handles[coord_source] = handle

legend_labels = [f"{source_name} coordinates" for source_name in legend_handles.keys()]
ax.legend(list(legend_handles.values()), legend_labels, loc='upper right', fontsize=10, framealpha=0.9)

ax.set_frame_on(False)

# DEM pre-process

In [ ]:
# download the needed rasters
dem_profile, dem = watershed_workflow.get_raster_on_shape(sources['DEM'], watershed.exterior(), crs)

if generate_plots:
    fig, axs = plt.subplots(1,1, figsize=(5,5))
    im1 = axs.imshow(dem, cmap='terrain')
    axs.set_title('unsmoothed DEM')
    fig.colorbar(im1, ax=axs, orientation='horizontal', pad=0.1, label='Elevation (Z)')
    

In [ ]:
dem_profile

## M2: convert raster crs first, then construct mesh from raster
M1 has been deprecated: use watershed_workflow.warp.xy to obtain mesh under daymet crs

In [ ]:
# download the needed rasters
dem_profile2, dem2 = watershed_workflow.get_raster_on_shape(sources['DEM'], watershed.exterior(), crs, out_crs=crs)

if generate_plots:
    # Calculate extent from the transform for geographic coordinates
    transform = dem_profile2['transform']
    rows, cols = dem2.shape
    left = transform.c
    right = transform.c + cols * transform.a
    top = transform.f
    bottom = transform.f + rows * transform.e
    extent = [left, right, bottom, top]
    
    fig, ax = plt.subplots(1,1, figsize=(5,5))
    im1 = ax.imshow(dem2, cmap='terrain', extent=extent, origin='upper')
    ax.set_title('DEM under Daymet crs')
    ax.set_xlabel('Easting (m)')
    ax.set_ylabel('Northing (m)')
    fig.colorbar(im1, ax=ax, orientation='horizontal', pad=0.1, label='Elevation (Z)')

In [ ]:
print(crs)
print(dem_profile2['crs'])

In [ ]:
output_path = "./data/dem/reprojected_dem.tif"
with rasterio.open(output_path, 'w', **dem_profile2) as dst:
    dst.write(dem2, 1)

In [ ]:
if generate_plots:
    # Calculate extent from the transform for geographic coordinates
    transform = dem_profile2['transform']
    rows, cols = dem2.shape
    left = transform.c
    right = transform.c + cols * transform.a
    top = transform.f
    bottom = transform.f + rows * transform.e
    extent = [left, right, bottom, top]

    fig, ax = plt.subplots(1, 1, figsize=(10, 10))
    im1 = ax.imshow(dem2, cmap='terrain', extent=extent, origin='upper')
    ax.set_xlabel('Easting (m)')
    ax.set_ylabel('Northing (m)')
    fig.colorbar(im1, ax=ax, orientation='horizontal', pad=0.1, label='Elevation (Z)')

    watershed_workflow.plot.hucs(watershed, crs, ax=ax, color='k', linewidth=0.75)
    watershed_workflow.plot.rivers(rivers, crs, ax=ax, color='red', linewidth=0.5)

    for site_id, site in sites.items():
        site_xy = shapely.geometry.Point(site['x'], site['y'])
        watershed_workflow.plot.shplys(site_xy, crs, ax=ax, color='blue', marker='o', markersize=100)
        ax.text(
            site['x'],
            site['y'],
            site['name'],
            fontsize=10,
            color='k',
            ha='right',
            va='bottom',
            transform=ax.transData,
        )

---
## CHECKPOINT: Site Selection

**IMPORTANT**: The cells below now process one site_id.

Set `selected_site_id` in the next cell. That cell will:
- select one site from the loaded observation-site table
- collect all valid source coordinates available for that site from `lin`, `delgado`, and `barnes`
- keep those valid coordinates in `category_points` for the downstream workflow

The zoom-in plot below uses the same `dx` and `dy` as before and visualizes all valid coordinates for the selected site.

---

In [ ]:
# ==============================================================================
# SELECT ONE SITE ID
# ==============================================================================
# Default: use the site_name from config.json as the selected site_id
selected_site_id = site_name

# Optional manual override:
# selected_site_id = "T05P"

print(f"Selecting site_id: {selected_site_id}")

available_site_ids = sorted(df_points["site_id"].astype(str).unique())
if selected_site_id not in available_site_ids:
    raise ValueError(
        f"Invalid site_id '{selected_site_id}'. "
        f"Must be one of: {available_site_ids}"
    )

selected_site = df_points.loc[df_points["site_id"] == selected_site_id].iloc[0]

# Keep this for compatibility with later cells that still reference selected_category
selected_category = selected_site_id

print("=" * 80)
print(f"SELECTED SITE ID: {selected_site_id}")
print("=" * 80)

source_order = ["lin", "delgado", "barnes"]

# Store all valid coordinates for the selected site in a downstream-compatible dict
category_points = {}
point_counter = 1

for source_name in source_order:
    lon = selected_site[f"{source_name}_lon"]
    lat = selected_site[f"{source_name}_lat"]
    x = selected_site[f"{source_name}_x"]
    y = selected_site[f"{source_name}_y"]

    if pd.notna(lon) and pd.notna(lat) and pd.notna(x) and pd.notna(y):
        category_points[point_counter] = {
            "point_id": point_counter,
            "source_name": source_name,
            "longitude": lon,
            "latitude": lat,
            "x": x,
            "y": y,
            "site_id": selected_site_id,
        }
        point_counter += 1

if len(category_points) == 0:
    raise ValueError(f"No valid coordinates found for site_id '{selected_site_id}'")

print(f"Found {len(category_points)} valid coordinate set(s) for site_id '{selected_site_id}'")
print()
print(f"{'Point ID':<10} {'Source':<10} {'Longitude':<12} {'Latitude':<12} {'X [m]':<15} {'Y [m]':<15}")
print(f"{'-'*10} {'-'*10} {'-'*12} {'-'*12} {'-'*15} {'-'*15}")

for point_id, point_info in category_points.items():
    print(
        f"{'P' + str(point_id):<10} "
        f"{point_info['source_name']:<10} "
        f"{point_info['longitude']:<12.6f} "
        f"{point_info['latitude']:<12.6f} "
        f"{point_info['x']:<15.2f} "
        f"{point_info['y']:<15.2f}"
    )

print()
print(f"Available point IDs: {sorted(category_points.keys())}")

In [ ]:
# zoom in to one selected site - showing all valid coordinates for that site
if generate_plots:
    # Get first valid coordinate as reference for zoom window
    first_point = list(category_points.values())[0]
    x_ref = first_point["x"]
    y_ref = first_point["y"]

    dx = 5000 / 2
    dy = 4000 / 2
    xmin = x_ref - dx / 2
    xmax = x_ref + dx / 2
    ymin = y_ref - dy / 2
    ymax = y_ref + dy / 2

    transform = dem_profile2["transform"]
    rows, cols = dem2.shape
    left = transform.c
    right = transform.c + cols * transform.a
    top = transform.f
    bottom = transform.f + rows * transform.e
    extent = [left, right, bottom, top]

    fig, ax = plt.subplots(1, 1, figsize=(7.5, 6))
    im1 = ax.imshow(dem2, cmap="terrain", extent=extent, origin="upper")
    ax.set_xlabel("Easting (m)")
    ax.set_ylabel("Northing (m)")
    fig.colorbar(im1, ax=ax, orientation="horizontal", pad=0.1, label="Elevation (Z)")

    watershed_workflow.plot.hucs(watershed, crs, ax=ax, color="k", linewidth=1)
    watershed_workflow.plot.rivers(rivers, crs, ax=ax, color="red", linewidth=1)

    # Plot all valid coordinates for the selected site
    for point_id, point_info in category_points.items():
        point_xy = shapely.geometry.Point(point_info["x"], point_info["y"])
        watershed_workflow.plot.shplys(
            point_xy, crs, ax=ax, color="blue", marker="o", markersize=100
        )
        ax.text(
            point_info["x"],
            point_info["y"] - 100,
            point_info["source_name"],
            fontsize=8,
            color="k",
            ha="center",
            va="top",
        )

    ax.set_xlim(xmin, xmax)
    ax.set_ylim(ymin, ymax)

# Find 2D transect start/end points

input:
- site lat lon
- river network
- 3D mesh

## [skip] Input one point

In [ ]:
flag_test_1point = True  # Set to False to skip this single-point test

if flag_test_1point:
    from pysheds.grid import Grid
    import time
    
    # Get the first point from category_points
    first_point = list(category_points.values())[0]
    x1 = first_point['x']
    y1 = first_point['y']
    
    print(f"Testing with Point {first_point['point_id']}: ({x1:.2f}, {y1:.2f})")
    
    # Load and compute flow path
    start_time = time.time()
    grid = Grid.from_raster('./data/dem/reprojected_dem.tif')
    dem = grid.read_raster('./data/dem/reprojected_dem.tif')
    
    # **IMPORTANT: Condition the DEM before computing flow direction**
    # Fill pits in DEM
    pit_filled_dem = grid.fill_pits(dem)
    
    # Fill depressions in DEM
    flooded_dem = grid.fill_depressions(pit_filled_dem)
    
    # Resolve flats in DEM
    inflated_dem = grid.resolve_flats(flooded_dem)
    
    # Now compute flow direction on the conditioned DEM
    fdir = grid.flowdir(inflated_dem, out_name='dir')
    
    # Get nearest cell and delineate catchment
    col, row = grid.nearest_cell(x1, y1)
    catch = grid.catchment(x=col, y=row, fdir=fdir, xytype='index')
    
    elapsed = time.time() - start_time
    print(f"Flow path calculation took {elapsed:.3f} seconds")
    print(f"Catchment covers {catch.sum()} cells")
    
    # Extract catchment cell coordinates
    catch_rows, catch_cols = np.where(catch)
    affine = grid.viewfinder.affine
    
    # Convert all catchment cells to real-world coordinates
    flow_x = []
    flow_y = []
    flow_z = []
    for r, c in zip(catch_rows, catch_cols):
        x, y = affine * (c, r)  # Column first, then row
        flow_x.append(x)
        flow_y.append(y)
        flow_z.append(dem2[r, c])  # Get elevation from dem2
    
    flow_x = np.array(flow_x)
    flow_y = np.array(flow_y)
    flow_z = np.array(flow_z)
    
    # Find the start point (highest elevation in catchment = furthest upstream)
    max_elev_idx = np.argmax(flow_z)
    start_x = flow_x[max_elev_idx]
    start_y = flow_y[max_elev_idx]
    start_z = flow_z[max_elev_idx]
    
    # End point is (x1, y1) - the pour point
    end_x = x1
    end_y = y1
    
    # Calculate Euclidean distance
    distance = np.sqrt((start_x - end_x)**2 + (start_y - end_y)**2)
    
    print("\n" + "="*60)
    print("FLOW PATH ANALYSIS")
    print("="*60)
    print(f"\nStart Point (Highest Elevation in Catchment):")
    print(f"  X: {start_x:.2f} m")
    print(f"  Y: {start_y:.2f} m")
    print(f"  Z: {start_z:.2f} m")
    
    print(f"\nEnd Point (Pour Point):")
    print(f"  X: {end_x:.2f} m")
    print(f"  Y: {end_y:.2f} m")
    
    print(f"\nStraight-line Distance: {distance:.2f} m")
    print(f"Elevation Drop: {start_z - dem2[row, col]:.2f} m")
else:
    print("Single-point test skipped (flag_test_1point = False)")

In [ ]:
if flag_test_1point: 
    from shapely.geometry import Polygon, MultiPolygon
    from shapely.ops import unary_union
    
    # Extract catchment boundary using rasterio features
    catch_array = np.array(catch, dtype=np.uint8) if hasattr(catch, 'astype') else catch.astype(np.uint8)
    shapes = rasterio.features.shapes(catch_array, transform=dem_profile2['transform'])
    
    # Get the catchment boundary polygon
    catchment_polygons = []
    for geom, value in shapes:
        if value == 1:  # Catchment cells
            catchment_polygons.append(shapely.geometry.shape(geom))
    
    # Merge all polygons into one
    if catchment_polygons:
        catchment_boundary = unary_union(catchment_polygons)
    else:
        catchment_boundary = None

In [ ]:
# Visualize with catchment boundary
if flag_test_1point and generate_plots:
    dx = 2500/2
    dy = 2000/2
    xmin = x1 - dx/2
    xmax = x1 + dx/2
    ymin = y1 - dy/2
    ymax = y1 + dy/2
    
    transform = dem_profile2['transform']
    rows_dem, cols_dem = dem2.shape
    left = transform.c
    right = transform.c + cols_dem * transform.a
    top = transform.f
    bottom = transform.f + rows_dem * transform.e
    extent = [left, right, bottom, top]
    
    fig, ax = plt.subplots(1,1, figsize=(9,7.5))
    im1 = ax.imshow(dem2, cmap='terrain', extent=extent, origin='upper')
    ax.set_xlabel('Easting (m)')
    ax.set_ylabel('Northing (m)')
    fig.colorbar(im1, ax=ax, orientation='horizontal', pad=0.1, label='Elevation (Z)')

    watershed_workflow.plot.hucs(watershed, crs, ax=ax, color='k', linewidth=1)
    watershed_workflow.plot.rivers(rivers, crs, ax=ax, color='red', linewidth=1)
    
    # Plot catchment boundary
    if catchment_boundary:
        if isinstance(catchment_boundary, MultiPolygon):
            for poly in catchment_boundary.geoms:
                x_coords, y_coords = poly.exterior.xy
                ax.plot(x_coords, y_coords, 'cyan', linewidth=2, zorder=3)
        else:
            x_coords, y_coords = catchment_boundary.exterior.xy
            ax.plot(x_coords, y_coords, 'cyan', linewidth=2, label='Catchment Boundary', zorder=3)
    
    # Plot the flow path catchment area (filled)
    ax.scatter(flow_x, flow_y, c='lightblue', s=1, alpha=0.3, label='D8 Catchment Area', zorder=2)
    
    # Plot start point (highest elevation)
    ax.plot(start_x, start_y, 'g^', markersize=14, markeredgecolor='darkgreen', 
            markeredgewidth=2, label='Start (Highest Elev)', zorder=5)
    ax.text(start_x, start_y+100, f'Start\nZ={start_z:.0f}m', 
            fontsize=8, color='darkgreen', ha='center', va='bottom', 
            bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.7))
    
    # Plot end point (pour point)
    site1_xy = shapely.geometry.Point(x1, y1)
    watershed_workflow.plot.shplys(site1_xy, crs, ax=ax, color='blue', marker='o', 
                                   markersize=100, zorder=5)
    # ax.text(x1, y1-200, f'Pour Point', 
    #         fontsize=8, color='blue', ha='center', va='top',
    #         bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.7))
    
    # Draw line between start and end
    ax.plot([start_x, end_x], [start_y, end_y], 'r--', linewidth=2.5, 
            label=f'Distance: {distance:.0f} m', zorder=4)
    
    ax.set_xlim(xmin, xmax)
    ax.set_ylim(ymin, ymax)
    
    # Remove duplicate labels in legend
    handles, labels = ax.get_legend_handles_labels()
    by_label = dict(zip(labels, handles))
    ax.legend(by_label.values(), by_label.keys(), loc='upper right', fontsize=9)
    
    plt.title('D8 Flow Path Analysis with Catchment Boundary')

## [skip] Snap end point to stream

In [ ]:
if flag_test_1point:

    from pysheds.grid import Grid
    from shapely.geometry import Polygon, MultiPolygon
    from shapely.ops import unary_union
    from scipy import ndimage
    import time

    # Get the first point from category_points
    first_point = list(category_points.values())[0]
    x1 = first_point['x']
    y1 = first_point['y']
    
    # Load and compute flow path
    start_time = time.time()
    grid = Grid.from_raster('./data/dem/reprojected_dem.tif')
    dem = grid.read_raster('./data/dem/reprojected_dem.tif')
    
    # **Condition the DEM**
    print("Conditioning DEM...")
    pit_filled_dem = grid.fill_pits(dem)
    flooded_dem = grid.fill_depressions(pit_filled_dem)
    inflated_dem = grid.resolve_flats(flooded_dem)
    
    # **Compute flow direction**
    print("Computing flow direction...")
    fdir = grid.flowdir(inflated_dem, out_name='dir')
    
    # **Compute flow accumulation**
    print("Computing flow accumulation...")
    acc = grid.accumulation(fdir)
    
    # **Define stream cells (high accumulation)**
    acc_threshold = 100
    stream_mask = acc > acc_threshold
    
    # **Snap pour point to stream**
    print(f"Snapping point to stream (accumulation > {acc_threshold})...")
    x_snap, y_snap = grid.snap_to_mask(stream_mask, (x1, y1))
    
    print(f"Original point: ({x1:.2f}, {y1:.2f})")
    print(f"Snapped point:  ({x_snap:.2f}, {y_snap:.2f})")
    snap_distance = np.sqrt((x_snap - x1)**2 + (y_snap - y1)**2)
    print(f"Snap distance: {snap_distance:.2f} m")
    
    # **Delineate catchment**
    catch = grid.catchment(x=x_snap, y=y_snap, fdir=fdir, xytype='coordinate')
    
    elapsed = time.time() - start_time
    print(f"Flow path calculation took {elapsed:.3f} seconds")
    print(f"Catchment covers {catch.sum()} cells")
    
    # Convert catch to numpy array
    if hasattr(catch, 'astype'):
        catch_array = np.array(catch, dtype=np.uint8)
    else:
        catch_array = catch.astype(np.uint8)
    
    affine = grid.viewfinder.affine
    
    # **CRITICAL: Identify stream cells within catchment**
    stream_in_catchment = stream_mask & catch_array
    
    # Get stream cell coordinates
    stream_rows, stream_cols = np.where(stream_in_catchment)
    print(f"Found {len(stream_rows)} stream cells in catchment")
    
    # **Find hillslope cells (non-stream cells in catchment)**
    hillslope_mask = catch_array & ~stream_in_catchment
    hillslope_rows, hillslope_cols = np.where(hillslope_mask)
    
    # **Find boundary cells**
    edge_mask = ndimage.binary_erosion(catch_array) != catch_array
    edge_catchment = catch_array & edge_mask
    edge_rows, edge_cols = np.where(edge_catchment)

In [ ]:
# Visualize stream, hillslope, and boundary cells
if flag_test_1point and generate_plots:
    dx = 4000/2
    dy = 3200/2
    xmin = x1 - dx/2*1.5
    xmax = x1 + dx/2
    ymin = y1 - dy/2
    ymax = y1 + dy/2
    
    transform = dem_profile2['transform']
    rows_dem, cols_dem = dem2.shape
    left = transform.c
    right = transform.c + cols_dem * transform.a
    top = transform.f
    bottom = transform.f + rows_dem * transform.e
    extent = [left, right, bottom, top]
    
    fig, ax = plt.subplots(1,1, figsize=(12,10))
    im1 = ax.imshow(dem2, cmap='terrain', extent=extent, origin='upper', alpha=0.6)
    ax.set_xlabel('Easting (m)')
    ax.set_ylabel('Northing (m)')
    fig.colorbar(im1, ax=ax, orientation='horizontal', pad=0.1, label='Elevation (Z)')

    watershed_workflow.plot.hucs(watershed, crs, ax=ax, color='k', linewidth=1)
    watershed_workflow.plot.rivers(rivers, crs, ax=ax, color='darkred', linewidth=1.5, label='NHD Rivers')
    
    # Convert cell indices to coordinates for plotting
    # Stream cells
    stream_x = []
    stream_y = []
    for r, c in zip(stream_rows, stream_cols):
        x, y = affine * (c, r)
        stream_x.append(x)
        stream_y.append(y)
    
    # Hillslope cells
    hillslope_x = []
    hillslope_y = []
    for r, c in zip(hillslope_rows, hillslope_cols):
        x, y = affine * (c, r)
        hillslope_x.append(x)
        hillslope_y.append(y)
    
    # Boundary cells
    boundary_x = []
    boundary_y = []
    for r, c in zip(edge_rows, edge_cols):
        x, y = affine * (c, r)
        boundary_x.append(x)
        boundary_y.append(y)
    
    # Plot the three cell types
    ax.scatter(hillslope_x, hillslope_y, c='k', s=3, alpha=0.3, 
               label=f'Hillslope Cells ({len(hillslope_rows)})', zorder=2)
    ax.scatter(stream_x, stream_y, c='blue', s=5, alpha=0.5, 
               label=f'Stream Cells (acc>{acc_threshold}) ({len(stream_rows)})', zorder=3)
    ax.scatter(boundary_x, boundary_y, c='red', s=8, alpha=0.7, 
               label=f'Boundary Cells ({len(edge_rows)})', zorder=4, marker='s')
    
    # # Plot catchment boundary
    # if catchment_boundary:
    #     if isinstance(catchment_boundary, MultiPolygon):
    #         for poly in catchment_boundary.geoms:
    #             x_coords, y_coords = poly.exterior.xy
    #             ax.plot(x_coords, y_coords, 'cyan', linewidth=2.5, zorder=5)
    #     else:
    #         x_coords, y_coords = catchment_boundary.exterior.xy
    #         ax.plot(x_coords, y_coords, 'cyan', linewidth=2.5, label='Catchment Boundary', zorder=5)
    
    # # # Plot original and snapped pour points
    # # ax.plot(x1, y1, 'rx', markersize=14, markeredgewidth=3, 
    # #         label='Original Pour Point', zorder=6)
    # ax.plot(x_snap, y_snap, 'mo', markersize=12, markerfacecolor='none',
    #         markeredgewidth=3, label='Snapped Pour Point', zorder=6)
    
    if snap_distance > 0:
        ax.plot([x1, x_snap], [y1, y_snap], 'm--', linewidth=2, alpha=0.7, zorder=5)
    
    ax.set_xlim(xmin, xmax)
    ax.set_ylim(ymin, ymax)
    
    # Legend
    handles, labels = ax.get_legend_handles_labels()
    by_label = dict(zip(labels, handles))
    ax.legend(by_label.values(), by_label.keys(), loc='upper right', fontsize=9, framealpha=0.95)
    
    plt.title(f'Catchment Cell Classification\nStream threshold: accumulation > {acc_threshold}')
    plt.tight_layout()

In [ ]:
# Approach 3: Analyze ALL neighbor cells with flexible center point
from scipy import ndimage

def analyze_all_neighbors(x_snap, y_snap, stream_mask, fdir, acc, dem2, affine, grid, center_offset=(0, 0)):
    """
    Analyze all 8 neighbors regardless of catchment boundary.
    Use catchment size to infer upstream/downstream relationships.
    
    Parameters:
    -----------
    center_offset : tuple of (row_offset, col_offset)
        Offset from snapped pour point to use as analysis center.
        Default (0, 0) uses the pour point itself.
        Example: (-1, 1) moves one row up (north) and one col right (east)
    """
    
    # Get the snapped pour point cell
    snap_col, snap_row = grid.nearest_cell(x_snap, y_snap)
    
    # Apply center offset to get the actual center point for analysis
    center_row = snap_row + center_offset[0]
    center_col = snap_col + center_offset[1]
    
    print(f"Analyzing center point at row={center_row}, col={center_col}")
    print(f"  (Pour point: row={snap_row}, col={snap_col}, offset={center_offset})")
    
    # Get coordinates of center point
    center_x, center_y = affine * (center_col, center_row)
    center_z = dem2[center_row, center_col]
    center_is_stream = stream_mask[center_row, center_col]
    
    print(f"  Center point: X={center_x:.2f}, Y={center_y:.2f}, Z={center_z:.1f}m")
    print(f"  Is stream cell: {center_is_stream}")
    
    # Compute catchment from center point
    catch_center = grid.catchment(x=center_col, y=center_row, fdir=fdir, xytype='index')
    catch_center_array = np.array(catch_center, dtype=np.uint8) if hasattr(catch_center, 'astype') else catch_center.astype(np.uint8)
    center_catchment_size = catch_center_array.sum()
    
    print(f"  Center point catchment size: {center_catchment_size} cells")
    
    # Define 8-neighbor offsets (row, col)
    neighbor_offsets = [
        (-1, -1), (-1, 0), (-1, 1),  # Top row
        (0, -1),           (0, 1),    # Middle row
        (1, -1),  (1, 0),  (1, 1)     # Bottom row
    ]
    
    # Analyze each neighbor
    neighbor_analysis = []
    
    for idx, (row_offset, col_offset) in enumerate(neighbor_offsets):
        neighbor_row = center_row + row_offset
        neighbor_col = center_col + col_offset
        
        # Check bounds
        if (0 <= neighbor_row < fdir.shape[0] and 0 <= neighbor_col < fdir.shape[1]):
            # Get coordinates
            neighbor_x, neighbor_y = affine * (neighbor_col, neighbor_row)
            
            # Get properties
            neighbor_z = dem2[neighbor_row, neighbor_col]
            neighbor_acc = acc[neighbor_row, neighbor_col]
            is_stream = stream_mask[neighbor_row, neighbor_col]
            
            # Compute neighbor's catchment
            catch_neighbor = grid.catchment(x=neighbor_col, y=neighbor_row, fdir=fdir, xytype='index')
            catch_neighbor_array = np.array(catch_neighbor, dtype=np.uint8) if hasattr(catch_neighbor, 'astype') else catch_neighbor.astype(np.uint8)
            neighbor_catchment_size = catch_neighbor_array.sum()
            
            # Check if in center point's catchment
            in_center_catchment = catch_center_array[neighbor_row, neighbor_col] == 1
            
            # Classify the neighbor
            if neighbor_catchment_size > center_catchment_size:
                cell_type = "Downstream"
            elif is_stream:
                cell_type = "Stream (Upstream)"
            elif in_center_catchment:
                cell_type = "Hillslope (Inside)"
            else:
                cell_type = "Hillslope (Outside)"
            
            neighbor_analysis.append({
                'idx': idx,
                'row': neighbor_row,
                'col': neighbor_col,
                'x': neighbor_x,
                'y': neighbor_y,
                'z': neighbor_z,
                'acc': neighbor_acc,
                'is_stream': is_stream,
                'in_center_catchment': in_center_catchment,
                'catchment_size': neighbor_catchment_size,
                'cell_type': cell_type,
                'catchment_array': catch_neighbor_array
            })
    
    return neighbor_analysis, center_catchment_size, catch_center_array, (center_x, center_y, center_z)




In [ ]:
if flag_test_1point:
    # =============================================================================
    # CONFIGURATION: Set center_offset here
    # =============================================================================
    # Options based on initial analysis:
    #   (0, 0)   - Use snapped pour point as center (default)
    #   (-1, 1)  - If upstream cell is at (-1, 1) relative to pour point
    #   (0, -1)  - If upstream cell is at (0, -1) relative to pour point
    #   (1, 0)   - Try downstream direction
    #   Custom   - Set any (row_offset, col_offset) you want to explore
    
    center_offset = (0, 0)  # MODIFY THIS to explore different stream cells
    
    # =============================================================================
    
    # Run the analysis
    neighbor_analysis, center_size, catch_center, center_coords = analyze_all_neighbors(
        x_snap, y_snap, stream_mask, fdir, acc, dem2, affine, grid, center_offset=center_offset
    )
    
    # Print results
    print("\n" + "="*100)
    print(f"NEIGHBOR CELL ANALYSIS (Center offset: {center_offset})")
    print("="*100)
    print(f"{'Idx':<5} {'Type':<22} {'In Center':<10} {'Elev (m)':<10} {'Acc':<10} {'Catch Size':<12} {'vs Center':<12}")
    print("-"*100)
    
    for nb in neighbor_analysis:
        size_ratio = f"{nb['catchment_size']/center_size:.2f}x"
        print(f"{nb['idx']:<5} {nb['cell_type']:<22} {str(nb['in_center_catchment']):<10} "
              f"{nb['z']:<10.1f} {nb['acc']:<10.0f} {nb['catchment_size']:<12} {size_ratio:<12}")
    
    # Identify valid hillslope neighbors
    hillslope_neighbors_all = [nb for nb in neighbor_analysis 
                              if 'Hillslope' in nb['cell_type']]
    
    print(f"\n✓ Found {len(hillslope_neighbors_all)} hillslope neighbors (all types)")
    for nb in hillslope_neighbors_all:
        print(f"  - Neighbor {nb['idx']}: {nb['cell_type']}, catchment={nb['catchment_size']} cells")
    
    # Suggest which upstream cells to try next
    upstream_stream_neighbors = [nb for nb in neighbor_analysis if nb['cell_type'] == "Stream (Upstream)"]
    if upstream_stream_neighbors:
        print(f"\n💡 SUGGESTION: Found {len(upstream_stream_neighbors)} upstream stream cell(s) to try as center:")
        for nb in upstream_stream_neighbors:
            # Calculate offset from original pour point
            snap_col, snap_row = grid.nearest_cell(x_snap, y_snap)
            offset = (nb['row'] - snap_row, nb['col'] - snap_col)
            print(f"  - Neighbor {nb['idx']}: Set center_offset = {offset}")

In [ ]:
if flag_test_1point:
    
    # Auto-select ALL hillslope neighbors (both inside and outside catchment)
    selected_neighbor_indices = [nb['idx'] for nb in neighbor_analysis if 'Hillslope' in nb['cell_type']]
    
    # Manual override option - uncomment and modify if needed:
    # selected_neighbor_indices = [0,1,4,5,6,7]  # MODIFY THIS as needed
    
    # Filter selected neighbors
    selected_neighbors = [nb for nb in neighbor_analysis if nb['idx'] in selected_neighbor_indices]
    
    print(f"\n✓ Auto-selected {len(selected_neighbors)} hillslope neighbors:")
    for nb in selected_neighbors:
        print(f"  - Neighbor {nb['idx']}: {nb['cell_type']}, catchment={nb['catchment_size']} cells")
    
    if len(selected_neighbors) == 0:
        print("\n⚠️  WARNING: No hillslope neighbors found!")
        print("   You may need to:")
        print("   Manually select neighbor indices by uncommenting the override line above")

In [ ]:
# # Visualize SELECTED neighbor subcatchments
# if flag_test_1point and generate_plots:
#     dx = 2000/2
#     dy = 1600/2
#     xmin = x1 - dx/2*1
#     xmax = x1 + dx/2*1
#     ymin = y1 - dy/2*1
#     ymax = y1 + dy/2*1
    
#     transform = dem_profile2['transform']
#     rows_dem, cols_dem = dem2.shape
#     left = transform.c
#     right = transform.c + cols_dem * transform.a
#     top = transform.f
#     bottom = transform.f + rows_dem * transform.e
#     extent = [left, right, bottom, top]
    
#     fig, ax = plt.subplots(1,1, figsize=(14,12))
#     im1 = ax.imshow(dem2, cmap='terrain', extent=extent, origin='upper', alpha=1.0)
#     ax.set_xlabel('Easting (m)')
#     ax.set_ylabel('Northing (m)')
#     fig.colorbar(im1, ax=ax, orientation='horizontal', pad=0.1, label='Elevation (Z)')

#     watershed_workflow.plot.hucs(watershed, crs, ax=ax, color='k', linewidth=1)
#     watershed_workflow.plot.rivers(rivers, crs, ax=ax, color='darkred', linewidth=1.5, label='NHD Rivers')
    
#     # Plot pour point catchment boundary
#     shapes = rasterio.features.shapes(catch_array, transform=dem_profile2['transform'])
#     catchment_polygons = []
#     for geom, value in shapes:
#         if value == 1:
#             catchment_polygons.append(shapely.geometry.shape(geom))
    
#     if catchment_polygons:
#         from shapely.ops import unary_union
#         catchment_boundary = unary_union(catchment_polygons)
#         if isinstance(catchment_boundary, MultiPolygon):
#             for poly in catchment_boundary.geoms:
#                 x_coords, y_coords = poly.exterior.xy
#                 ax.plot(x_coords, y_coords, 'cyan', linewidth=2.5, zorder=3)
#         else:
#             x_coords, y_coords = catchment_boundary.exterior.xy
#             ax.plot(x_coords, y_coords, 'cyan', linewidth=2.5, label='Pour Point Catchment', zorder=3)
    
#     # Define colors with better contrast to green DEM
#     colors_list = ['darkviolet', 'deeppink', 'orangered', 'gold', 'hotpink', 'coral', 'yellow']
#     markers_list = ['s'] #['o', 's', '^', 'v', 'D', 'P', '*', 'X']
    
#     # Plot SELECTED neighbors' subcatchments
#     for i, nb in enumerate(selected_neighbors):
#         subcatch_rows, subcatch_cols = np.where(nb['catchment_array'])
        
#         subcatch_x = []
#         subcatch_y = []
#         for r, c in zip(subcatch_rows, subcatch_cols):
#             x, y = affine * (c, r)
#             subcatch_x.append(x)
#             subcatch_y.append(y)
        
#         color = colors_list[i % len(colors_list)]
#         marker = markers_list[i % len(markers_list)]
        
#         # Plot subcatchment area
#         ax.scatter(subcatch_x, subcatch_y, c=color, s=25, alpha=1.0, 
#                   marker=marker, label=f'Hillslope {i+1} (N{nb["idx"]}, {nb["catchment_size"]} cells)', 
#                   zorder=3, edgecolors='none')
        
#         # Mark the neighbor cell with smaller X marker (no text box)
#         ax.plot(nb['x'], nb['y'], marker='x', markersize=8, 
#                markerfacecolor=color, markeredgecolor='black', markeredgewidth=2, zorder=6)
    
#     # Plot ALL neighbor cells (not selected) with smaller gray X
#     for nb in neighbor_analysis:
#         if nb['idx'] not in selected_neighbor_indices:
#             ax.plot(nb['x'], nb['y'], 'x', markersize=8, color='gray', 
#                    markeredgewidth=1.5, alpha=1.0, zorder=5)
    
#     # Plot pour point with smaller marker
#     ax.plot(x_snap, y_snap, 'mo', markersize=5, markerfacecolor='magenta',
#             markeredgecolor='darkmagenta', markeredgewidth=2, label='Pour Point', zorder=7)
    
#     ax.set_xlim(xmin, xmax)
#     ax.set_ylim(ymin, ymax)
    
#     # Legend
#     handles, labels = ax.get_legend_handles_labels()
#     by_label = dict(zip(labels, handles))
#     ax.legend(by_label.values(), by_label.keys(), loc='upper left', fontsize=8, 
#              framealpha=0.95, ncol=1)
    
#     plt.title(f'Selected Hillslope Subcatchments\n{len(selected_neighbors)} hillslopes selected')
#     plt.tight_layout()

In [ ]:
if flag_test_1point:
    # Compute transect endpoints for each selected subcatchment
    print("\n" + "="*100)
    print("TRANSECT ANALYSIS FOR SELECTED HILLSLOPES")
    print("="*100)
    print(f"{'Hillslope':<12} {'Neighbor':<10} {'Start X':<12} {'Start Y':<12} {'Start Z':<10} "
          f"{'End X':<12} {'End Y':<12} {'End Z':<10} {'Distance':<12} {'Elev Drop':<12}")
    print("-"*100)
    
    transect_info = []
    
    # Compute transect endpoints - REVISED to prioritize longest distance
    for i, nb in enumerate(selected_neighbors):
        subcatch_rows, subcatch_cols = np.where(nb['catchment_array'])
        
        # Calculate distances from all cells to pour point
        distances_to_pour = []
        elevations = []
        
        for r, c in zip(subcatch_rows, subcatch_cols):
            x, y = affine * (c, r)
            dist = np.sqrt((x - x_snap)**2 + (y - y_snap)**2)
            z = dem2[r, c]
            distances_to_pour.append(dist)
            elevations.append(z)
        
        distances_to_pour = np.array(distances_to_pour)
        elevations = np.array(elevations)
        
        # PRIMARY: Find the farthest point
        max_dist_idx = np.argmax(distances_to_pour)
        
        # Optional sanity check: ensure it's reasonably high elevation
        # (e.g., within top 10% of elevations)
        elev_threshold = np.percentile(elevations, 90)
        
        if elevations[max_dist_idx] < elev_threshold:
            print(f"  [WARNING] H{i+1}: Farthest point (Z={elevations[max_dist_idx]:.1f}m) "
                  f"is below 90th percentile (Z={elev_threshold:.1f}m)")
        
        # Use the farthest point
        start_row = subcatch_rows[max_dist_idx]
        start_col = subcatch_cols[max_dist_idx]
        start_x, start_y = affine * (start_col, start_row)
        start_z = elevations[max_dist_idx]
        
        # End point coordinates
        end_x = x_snap
        end_y = y_snap
        end_col, end_row = grid.nearest_cell(x_snap, y_snap)
        end_z = dem2[end_row, end_col]
        
        # Calculate distance and elevation drop
        distance = distances_to_pour[max_dist_idx]
        elev_drop = start_z - end_z
        
        # Store information
        transect_info.append({
            'hillslope_id': i + 1,
            'neighbor_idx': nb['idx'],
            'start_coords': (start_x, start_y),
            'start_z': start_z,
            'end_coords': (end_x, end_y),
            'end_z': end_z,
            'distance': distance,
            'elev_drop': elev_drop
        })
        
        # Print results
        print(f"{'Hillslope ' + str(i+1):<12} {'N' + str(nb['idx']):<10} "
              f"{start_x:<12.2f} {start_y:<12.2f} {start_z:<10.1f} "
              f"{end_x:<12.2f} {end_y:<12.2f} {end_z:<10.1f} "
              f"{distance:<12.1f} {elev_drop:<12.1f}")
        
        # print(f"  [DEBUG] H{i+1}: Selected point at distance={distance:.1f}m, "
        #       f"Z={start_z:.1f}m (max_Z={np.max(elevations):.1f}m)")
    
    # Summary statistics
    print("\n" + "="*100)
    print("SUMMARY STATISTICS")
    print("="*100)
    
    if len(transect_info) > 0:
        distances = [t['distance'] for t in transect_info]
        elev_drops = [t['elev_drop'] for t in transect_info]
        
        print(f"Distance range: {np.min(distances):.1f} - {np.max(distances):.1f} m (mean: {np.mean(distances):.1f} m)")
        print(f"Elevation drop range: {np.min(elev_drops):.1f} - {np.max(elev_drops):.1f} m (mean: {np.mean(elev_drops):.1f} m)")
        print(f"Average slope: {np.mean([ed/d for ed, d in zip(elev_drops, distances)]):.3f} (mean gradient)")
    else:
        print("No transects computed.")

In [ ]:
# Visualize SELECTED neighbor subcatchments with transect lines and start points
if flag_test_1point and generate_plots:
    # # Manual configuration (commented out)
    # dx = 2000/2
    # dy = 1600/2
    # xmin = x1 - dx/2*1
    # xmax = x1 + dx/2*1
    # ymin = y1 - dy/2*1
    # ymax = y1 + dy/2*1
    
    # Automatically determine plot bounds from transect coordinates
    all_x = [x_snap]  # Start with pour point
    all_y = [y_snap]
    
    for transect in transect_info:
        start_x, start_y = transect['start_coords']
        all_x.append(start_x)
        all_y.append(start_y)
    
    # Calculate bounds with 50% buffer
    x_range = max(all_x) - min(all_x)
    y_range = max(all_y) - min(all_y)
    buffer_x = x_range * 1.0
    buffer_y = y_range * 1.0
    
    xmin = min(all_x) - buffer_x
    xmax = max(all_x) + buffer_x
    ymin = min(all_y) - buffer_y
    ymax = max(all_y) + buffer_y
    
    transform = dem_profile2['transform']
    rows_dem, cols_dem = dem2.shape
    left = transform.c
    right = transform.c + cols_dem * transform.a
    top = transform.f
    bottom = transform.f + rows_dem * transform.e
    extent = [left, right, bottom, top]
    
    fig, ax = plt.subplots(1,1, figsize=(14,12))
    im1 = ax.imshow(dem2, cmap='terrain', extent=extent, origin='upper', alpha=1.0)
    ax.set_xlabel('Easting (m)')
    ax.set_ylabel('Northing (m)')
    fig.colorbar(im1, ax=ax, orientation='horizontal', pad=0.1, label='Elevation (Z)')

    watershed_workflow.plot.hucs(watershed, crs, ax=ax, color='k', linewidth=1)
    watershed_workflow.plot.rivers(rivers, crs, ax=ax, color='darkred', linewidth=1.5, label='NHD Rivers')
    
    # Plot pour point catchment boundary
    shapes = rasterio.features.shapes(catch_array, transform=dem_profile2['transform'])
    catchment_polygons = []
    for geom, value in shapes:
        if value == 1:
            catchment_polygons.append(shapely.geometry.shape(geom))
    
    if catchment_polygons:
        from shapely.ops import unary_union
        catchment_boundary = unary_union(catchment_polygons)
        if isinstance(catchment_boundary, MultiPolygon):
            for poly in catchment_boundary.geoms:
                x_coords, y_coords = poly.exterior.xy
                ax.plot(x_coords, y_coords, 'cyan', linewidth=2.5, zorder=3)
        else:
            x_coords, y_coords = catchment_boundary.exterior.xy
            ax.plot(x_coords, y_coords, 'cyan', linewidth=2.5, label='Pour Point Catchment', zorder=3)
    
    # Define colors with better contrast
    colors_list = ['darkviolet', 'deeppink', 'orangered', 'gold', 'hotpink', 'coral', 'yellow']
    markers_list = ['s']
    
    # Plot SELECTED neighbors' subcatchments with transect lines
    for i, nb in enumerate(selected_neighbors):
        subcatch_rows, subcatch_cols = np.where(nb['catchment_array'])
        
        subcatch_x = []
        subcatch_y = []
        for r, c in zip(subcatch_rows, subcatch_cols):
            x, y = affine * (c, r)
            subcatch_x.append(x)
            subcatch_y.append(y)
        
        color = colors_list[i % len(colors_list)]
        marker = markers_list[i % len(markers_list)]
        
        # Plot subcatchment area
        ax.scatter(subcatch_x, subcatch_y, c=color, s=25, alpha=0.6, 
                  marker=marker, label=f'Hillslope {i+1} (N{nb["idx"]}, {nb["catchment_size"]} cells)', 
                  zorder=3, edgecolors='none')
        
        # Get transect info for this hillslope
        transect = transect_info[i]
        start_x, start_y = transect['start_coords']
        end_x, end_y = transect['end_coords']
        
        # Draw transect line
        ax.plot([start_x, end_x], [start_y, end_y], color=color, linewidth=3, 
                linestyle='-', alpha=0.9, zorder=5)
        
        # Plot start point with larger marker
        ax.plot(start_x, start_y, '^', color=color, markersize=12, 
                markeredgecolor='black', markeredgewidth=2, zorder=6)
        
        # Add label at start point only if distance > 100m
        if transect["distance"] > 100:
            ax.text(start_x, start_y + 30, f'H{i+1}\nd={transect["distance"]:.0f}m', 
                    fontsize=8, color='black', ha='center', va='bottom',
                    bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.8, edgecolor=color),
                    zorder=7)
    
    # Plot ALL neighbor cells (not selected) with smaller gray X
    for nb in neighbor_analysis:
        if nb['idx'] not in selected_neighbor_indices:
            ax.plot(nb['x'], nb['y'], 'x', markersize=8, color='gray', 
                   markeredgewidth=1.5, alpha=1.0, zorder=5)
    
    # Plot pour point with larger marker
    ax.plot(x_snap, y_snap, 'mo', markersize=10, markerfacecolor='magenta',
            markeredgecolor='darkmagenta', markeredgewidth=2.5, label='Pour Point', zorder=8)
    
    ax.set_xlim(xmin, xmax)
    ax.set_ylim(ymin, ymax)
    
    # Legend
    handles, labels = ax.get_legend_handles_labels()
    by_label = dict(zip(labels, handles))
    ax.legend(by_label.values(), by_label.keys(), loc='upper left', fontsize=8, 
             framealpha=0.95, ncol=1)
    
    plt.title(f'Hillslope Transects with Start/End Points\n{len(selected_neighbors)} hillslopes, max distance selection')
    plt.tight_layout()

## [formal] loop points and find qualified hillslope

In [ ]:
flag_test_allpoints = False  # Set to False to skip processing all points

if flag_test_allpoints:
    from pysheds.grid import Grid
    from shapely.geometry import Polygon, MultiPolygon
    from shapely.ops import unary_union
    from scipy import ndimage
    import time
    
    print("="*100)
    print("PROCESSING ALL POINTS IN SELECTED CATEGORY")
    print("="*100)
    
    # Load grid once
    grid = Grid.from_raster('./data/dem/reprojected_dem.tif')
    dem = grid.read_raster('./data/dem/reprojected_dem.tif')
    
    # Condition the DEM once
    print("\nConditioning DEM...")
    pit_filled_dem = grid.fill_pits(dem)
    flooded_dem = grid.fill_depressions(pit_filled_dem)
    inflated_dem = grid.resolve_flats(flooded_dem)
    
    # Compute flow direction and accumulation once
    print("Computing flow direction and accumulation...")
    fdir = grid.flowdir(inflated_dem, out_name='dir')
    acc = grid.accumulation(fdir)
    
    # Define stream cells once
    acc_threshold = 100
    stream_mask = acc > acc_threshold
    
    affine = grid.viewfinder.affine
    
    # Step 1: Snap all pour points to stream
    print("\n" + "="*100)
    print("STEP 1: SNAPPING ALL POUR POINTS TO STREAM")
    print("="*100)
    
    snapped_points = {}
    snap_groups = {}  # Group points by snapped location
    
    for point_id, point_info in category_points.items():
        x_orig = point_info['x']
        y_orig = point_info['y']
        
        # Snap to stream
        x_snap, y_snap = grid.snap_to_mask(stream_mask, (x_orig, y_orig))
        snap_distance = np.sqrt((x_snap - x_orig)**2 + (y_snap - y_orig)**2)
        
        # Store snapped info
        snapped_points[point_id] = {
            'x_orig': x_orig,
            'y_orig': y_orig,
            'x_snap': x_snap,
            'y_snap': y_snap,
            'snap_distance': snap_distance
        }
        
        # Group by snapped location (round to avoid floating point issues)
        snap_key = (round(x_snap, 2), round(y_snap, 2))
        if snap_key not in snap_groups:
            snap_groups[snap_key] = []
        snap_groups[snap_key].append(point_id)
        
        print(f"Point {point_id}: ({x_orig:.2f}, {y_orig:.2f}) -> ({x_snap:.2f}, {y_snap:.2f}), "
              f"snap_dist={snap_distance:.2f}m")
    
    # Report grouping
    print(f"\n✓ Total unique snapped locations: {len(snap_groups)}")
    for snap_key, point_ids in snap_groups.items():
        if len(point_ids) > 1:
            print(f"  Location {snap_key}: Points {point_ids} snap to same location")
    
    # Step 2: Process each unique snapped location
    print("\n" + "="*100)
    print("STEP 2: ANALYZING HILLSLOPES FOR EACH SNAPPED LOCATION")
    print("="*100)
    
    all_results = {}
    
    for snap_key, point_ids in snap_groups.items():
        x_snap, y_snap = snap_key
        
        print(f"\n{'='*80}")
        print(f"Processing snapped location: ({x_snap:.2f}, {y_snap:.2f})")
        print(f"  Corresponding to Point(s): {point_ids}")
        print(f"{'='*80}")
        
        # Delineate catchment
        catch = grid.catchment(x=x_snap, y=y_snap, fdir=fdir, xytype='coordinate')
        catch_array = np.array(catch, dtype=np.uint8) if hasattr(catch, 'astype') else catch.astype(np.uint8)
        
        print(f"  Catchment covers {catch_array.sum()} cells")
        
        # Analyze neighbors with center_offset = (0, 0)
        center_offset = (0, 0)
        neighbor_analysis, center_size, catch_center, center_coords = analyze_all_neighbors(
            x_snap, y_snap, stream_mask, fdir, acc, dem2, affine, grid, center_offset=center_offset
        )
        
        # Auto-select hillslope neighbors
        selected_neighbor_indices = [nb['idx'] for nb in neighbor_analysis if 'Hillslope' in nb['cell_type']]
        selected_neighbors = [nb for nb in neighbor_analysis if nb['idx'] in selected_neighbor_indices]
        
        print(f"\n  ✓ Found {len(selected_neighbors)} hillslope neighbors")
        
        # Compute transects for selected hillslopes
        transect_info = []
        
        for i, nb in enumerate(selected_neighbors):
            subcatch_rows, subcatch_cols = np.where(nb['catchment_array'])
            
            # Calculate distances from all cells to pour point
            distances_to_pour = []
            elevations = []
            
            for r, c in zip(subcatch_rows, subcatch_cols):
                x, y = affine * (c, r)
                dist = np.sqrt((x - x_snap)**2 + (y - y_snap)**2)
                z = dem2[r, c]
                distances_to_pour.append(dist)
                elevations.append(z)
            
            distances_to_pour = np.array(distances_to_pour)
            elevations = np.array(elevations)
            
            # Find the farthest point
            max_dist_idx = np.argmax(distances_to_pour)
            
            start_row = subcatch_rows[max_dist_idx]
            start_col = subcatch_cols[max_dist_idx]
            start_x, start_y = affine * (start_col, start_row)
            start_z = elevations[max_dist_idx]
            
            # End point coordinates
            end_x = x_snap
            end_y = y_snap
            end_col, end_row = grid.nearest_cell(x_snap, y_snap)
            end_z = dem2[end_row, end_col]
            
            # Calculate distance and elevation drop
            distance = distances_to_pour[max_dist_idx]
            elev_drop = start_z - end_z
            
            transect_info.append({
                'hillslope_id': i + 1,
                'neighbor_idx': nb['idx'],
                'start_coords': (start_x, start_y),
                'start_z': start_z,
                'end_coords': (end_x, end_y),
                'end_z': end_z,
                'distance': distance,
                'elev_drop': elev_drop
            })
        
        # Store results for this snapped location
        for point_id in point_ids:
            all_results[point_id] = {
                'x_snap': x_snap,
                'y_snap': y_snap,
                'catch_array': catch_array,
                'neighbor_analysis': neighbor_analysis,
                'selected_neighbors': selected_neighbors,
                'transect_info': transect_info
            }
        
        # Print summary
        if len(transect_info) > 0:
            distances = [t['distance'] for t in transect_info]
            long_transects = [t for t in transect_info if t['distance'] > 100]
            print(f"\n  Transect summary:")
            print(f"    Total hillslopes: {len(transect_info)}")
            print(f"    Hillslopes > 100m: {len(long_transects)}")
            print(f"    Distance range: {np.min(distances):.1f} - {np.max(distances):.1f} m")
    
    print("\n" + "="*100)
    print("✓ ANALYSIS COMPLETE FOR ALL POINTS")
    print("="*100)
else:
    print("All-points processing skipped (flag_test_allpoints = False)")

In [ ]:
# Visualize hillslopes for all points (only those > 100m)
if flag_test_allpoints and generate_plots:
    
    print("\n" + "="*100)
    print("GENERATING FIGURES FOR ALL POINTS")
    print("="*100)
    
    for point_id, results in all_results.items():
        x_snap = results['x_snap']
        y_snap = results['y_snap']
        catch_array = results['catch_array']
        selected_neighbors = results['selected_neighbors']
        transect_info = results['transect_info']
        
        # Filter transects > 100m
        long_transects = [t for t in transect_info if t['distance'] > 100]
        
        if len(long_transects) == 0:
            print(f"\nPoint {point_id}: No hillslopes > 100m, skipping plot")
            continue
        
        print(f"\nPoint {point_id}: Plotting {len(long_transects)} hillslopes > 100m")
        
        # Automatically determine plot bounds from transect coordinates
        all_x = [x_snap]
        all_y = [y_snap]
        
        for transect in long_transects:
            start_x, start_y = transect['start_coords']
            all_x.append(start_x)
            all_y.append(start_y)
        
        # Calculate bounds with 100% buffer
        x_range = max(all_x) - min(all_x)
        y_range = max(all_y) - min(all_y)
        buffer_x = x_range * 1.0
        buffer_y = y_range * 1.0
        
        xmin = min(all_x) - buffer_x
        xmax = max(all_x) + buffer_x
        ymin = min(all_y) - buffer_y
        ymax = max(all_y) + buffer_y
        
        # Setup plot
        transform = dem_profile2['transform']
        rows_dem, cols_dem = dem2.shape
        left = transform.c
        right = transform.c + cols_dem * transform.a
        top = transform.f
        bottom = transform.f + rows_dem * transform.e
        extent = [left, right, bottom, top]
        
        fig, ax = plt.subplots(1, 1, figsize=(14, 12))
        im1 = ax.imshow(dem2, cmap='terrain', extent=extent, origin='upper', alpha=1.0)
        ax.set_xlabel('Easting (m)')
        ax.set_ylabel('Northing (m)')
        fig.colorbar(im1, ax=ax, orientation='horizontal', pad=0.1, label='Elevation (Z)')
        
        watershed_workflow.plot.hucs(watershed, crs, ax=ax, color='k', linewidth=1)
        watershed_workflow.plot.rivers(rivers, crs, ax=ax, color='darkred', linewidth=1.5, label='NHD Rivers')
        
        # Plot catchment boundary
        shapes = rasterio.features.shapes(catch_array, transform=dem_profile2['transform'])
        catchment_polygons = []
        for geom, value in shapes:
            if value == 1:
                catchment_polygons.append(shapely.geometry.shape(geom))
        
        if catchment_polygons:
            catchment_boundary = unary_union(catchment_polygons)
            if isinstance(catchment_boundary, MultiPolygon):
                for poly in catchment_boundary.geoms:
                    x_coords, y_coords = poly.exterior.xy
                    ax.plot(x_coords, y_coords, 'cyan', linewidth=2.5, zorder=3)
            else:
                x_coords, y_coords = catchment_boundary.exterior.xy
                ax.plot(x_coords, y_coords, 'cyan', linewidth=2.5, label='Pour Point Catchment', zorder=3)
        
        # Define colors
        colors_list = ['darkviolet', 'deeppink', 'orangered', 'gold', 'hotpink', 'coral', 'yellow']
        
        # Plot only hillslopes > 100m
        plotted_count = 0
        for transect in long_transects:
            # Find corresponding neighbor
            matching_neighbors = [nb for nb in selected_neighbors 
                                 if nb['idx'] == transect['neighbor_idx']]
            if not matching_neighbors:
                continue
            
            nb = matching_neighbors[0]
            subcatch_rows, subcatch_cols = np.where(nb['catchment_array'])
            
            subcatch_x = []
            subcatch_y = []
            for r, c in zip(subcatch_rows, subcatch_cols):
                x, y = affine * (c, r)
                subcatch_x.append(x)
                subcatch_y.append(y)
            
            color = colors_list[plotted_count % len(colors_list)]
            
            # Plot subcatchment area
            ax.scatter(subcatch_x, subcatch_y, c=color, s=25, alpha=0.6, 
                      marker='s', 
                      label=f'Hillslope {transect["hillslope_id"]} (N{nb["idx"]}, {nb["catchment_size"]} cells)', 
                      zorder=3, edgecolors='none')
            
            # Get transect endpoints
            start_x, start_y = transect['start_coords']
            end_x, end_y = transect['end_coords']
            
            # Draw transect line
            ax.plot([start_x, end_x], [start_y, end_y], color=color, linewidth=3, 
                    linestyle='-', alpha=0.9, zorder=5)
            
            # Plot start point
            ax.plot(start_x, start_y, '^', color=color, markersize=12, 
                    markeredgecolor='black', markeredgewidth=2, zorder=6)
            
            # Add label
            ax.text(start_x, start_y + 30, f'H{transect["hillslope_id"]}\nd={transect["distance"]:.0f}m', 
                    fontsize=8, color='black', ha='center', va='bottom',
                    bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.8, edgecolor=color),
                    zorder=7)
            
            plotted_count += 1
        
        # Plot pour point
        ax.plot(x_snap, y_snap, 'mo', markersize=10, markerfacecolor='magenta',
                markeredgecolor='darkmagenta', markeredgewidth=2.5, label='Pour Point', zorder=8)
        
        ax.set_xlim(xmin, xmax)
        ax.set_ylim(ymin, ymax)
        
        # Legend
        handles, labels = ax.get_legend_handles_labels()
        by_label = dict(zip(labels, handles))
        ax.legend(by_label.values(), by_label.keys(), loc='upper left', fontsize=8, 
                 framealpha=0.95, ncol=1)
        
        plt.title(f'Point {point_id}: Hillslope Transects (> 100m)\n'
                  f'{len(long_transects)} hillslopes shown, Category: {selected_category}')
        plt.tight_layout()
        
        # Save figure
        output_filename = f'./images/{site_name}/hillslopes_point{point_id}_{selected_category}.png'
        fig.savefig(output_filename, dpi=300, bbox_inches='tight')
        print(f"  ✓ Saved: {output_filename}")
        
        plt.close(fig)
    
    print("\n✓ All figures generated")

## save start/end_coords to mat file

### save all hillslopes

In [ ]:
# Save ALL hillslopes > 100m automatically
if flag_test_allpoints:
    from scipy.io import savemat
    
    print("\n" + "="*100)
    print("SAVING ALL HILLSLOPES > 100m")
    print("="*100)
    
    all_transects_to_save = []
    
    # Iterate through all points and collect transects > 100m
    for point_id, results in all_results.items():
        transect_list = results['transect_info']
        
        # Filter transects > 100m
        long_transects = [t for t in transect_list if t['distance'] > 100]
        
        for transect in long_transects:
            # Generate automatic name
            auto_name = f'P{point_id}_H{transect["hillslope_id"]}'
            
            # Store transect data
            all_transects_to_save.append({
                'name': auto_name,
                'point_id': point_id,
                'hillslope_id': transect['hillslope_id'],
                'neighbor_idx': transect['neighbor_idx'],
                'start_coords': np.array(transect['start_coords']),
                'start_z': transect['start_z'],
                'end_coords': np.array(transect['end_coords']),
                'end_z': transect['end_z'],
                'distance': transect['distance'],
                'elev_drop': transect['elev_drop']
            })
    
    print(f"\n✓ Found {len(all_transects_to_save)} hillslopes > 100m across all points")
    
    if len(all_transects_to_save) > 0:
        # Prepare data for MATLAB format
        start_coords_array = np.array([t['start_coords'] for t in all_transects_to_save])
        end_coords_array = np.array([t['end_coords'] for t in all_transects_to_save])
        
        # Create metadata dictionary
        metadata = {
            'names': [t['name'] for t in all_transects_to_save],
            'point_ids': [t['point_id'] for t in all_transects_to_save],
            'hillslope_ids': [t['hillslope_id'] for t in all_transects_to_save],
            'neighbor_indices': [t['neighbor_idx'] for t in all_transects_to_save],
            'distances': [t['distance'] for t in all_transects_to_save],
            'elev_drops': [t['elev_drop'] for t in all_transects_to_save],
            'start_elevations': [t['start_z'] for t in all_transects_to_save],
            'end_elevations': [t['end_z'] for t in all_transects_to_save]
        }
        
        data_to_save = {
            'start_coords': start_coords_array,
            'end_coords': end_coords_array,
            'metadata': metadata,
            'num_transects': len(all_transects_to_save),
            'selected_category': selected_category
        }
        
        # Save to MAT file with "_all" suffix
        all_mat_filename = f'../data-processed/{site_name}/startendcoords_{site_name}_all_gt100m.mat'
        savemat(all_mat_filename, data_to_save)
        
        print(f"\n✓ Saved {len(all_transects_to_save)} transect(s) to: {all_mat_filename}")
        print(f"\nFile contents:")
        print(f"  - start_coords: [{len(all_transects_to_save)} x 2] array")
        print(f"  - end_coords: [{len(all_transects_to_save)} x 2] array")
        print(f"  - metadata: Dictionary with transect details")
        print(f"  - num_transects: {len(all_transects_to_save)}")
        print(f"  - selected_category: '{selected_category}'")
        
        # Save summary CSV
        csv_filename = f'../data-processed/{site_name}/startendcoords_{site_name}_all_gt100m_summary.csv'
        summary_df = pd.DataFrame({
            'transect_name': [t['name'] for t in all_transects_to_save],
            'point_id': [t['point_id'] for t in all_transects_to_save],
            'hillslope_id': [t['hillslope_id'] for t in all_transects_to_save],
            'neighbor_idx': [t['neighbor_idx'] for t in all_transects_to_save],
            'start_x': [t['start_coords'][0] for t in all_transects_to_save],
            'start_y': [t['start_coords'][1] for t in all_transects_to_save],
            'start_z': [t['start_z'] for t in all_transects_to_save],
            'end_x': [t['end_coords'][0] for t in all_transects_to_save],
            'end_y': [t['end_coords'][1] for t in all_transects_to_save],
            'end_z': [t['end_z'] for t in all_transects_to_save],
            'distance_m': [t['distance'] for t in all_transects_to_save],
            'elev_drop_m': [t['elev_drop'] for t in all_transects_to_save]
        })
        summary_df.to_csv(csv_filename, index=False)
        print(f"✓ Also saved summary CSV: {csv_filename}")
        
        # Print summary by point
        print(f"\n{'Point ID':<10} {'# Hillslopes':<15} {'Distance Range (m)':<25}")
        print("-"*50)
        for point_id in sorted(set(t['point_id'] for t in all_transects_to_save)):
            point_transects = [t for t in all_transects_to_save if t['point_id'] == point_id]
            distances = [t['distance'] for t in point_transects]
            print(f"{point_id:<10} {len(point_transects):<15} "
                  f"{np.min(distances):.1f} - {np.max(distances):.1f}")
    else:
        print("\n⚠️  No hillslopes > 100m found in the analysis")
else:
    print("Skipped: flag_test_allpoints = False")

In [ ]:
transect_info

### save user selected hillslopes

- **CAUTION**, better after 0b-checkBChead.ipynb

In [ ]:
# ==============================================================================
# USER INPUT: Select hillslopes from transect_info
# ==============================================================================
#
# Format: List of dictionaries, each containing:
#   - 'hillslope_id': The hillslope ID from transect_info
#   - 'name': A descriptive name for this transect (optional)
#
# Example:
# selected_hillslopes = [
#     {'hillslope_id': 2, 'name': 'P_H2'},
# ]

selected_hillslopes = [
    {'point_id': 1,'hillslope_id': 2, 'name': 'P_H2'},
]

# ==============================================================================

print("\n" + "=" * 100)
print("SAVING SELECTED HILLSLOPE TRANSECTS")
print("=" * 100)

# Validate and collect transect data
transects_to_save = []

available_ids = [t['hillslope_id'] for t in transect_info]

for idx, selection in enumerate(selected_hillslopes):
    hillslope_id = selection['hillslope_id']
    custom_name = selection.get('name', f'H{hillslope_id}')

    print(f"\n[{idx+1}] Processing selection: Hillslope {hillslope_id}")

    matching_transects = [t for t in transect_info if t['hillslope_id'] == hillslope_id]

    if len(matching_transects) == 0:
        print(f"  ✗ ERROR: Hillslope {hillslope_id} not found. Available: {available_ids}")
        continue

    transect = matching_transects[0]

    start_coords = np.array(transect['start_coords'])
    start_z = transect['start_z']
    end_coords = np.array(transect['end_coords'])
    end_z = transect['end_z']
    distance = transect['distance']
    elev_drop = transect['elev_drop']
    neighbor_idx = transect['neighbor_idx']

    transects_to_save.append({
        'name': custom_name,
        'hillslope_id': hillslope_id,
        'neighbor_idx': neighbor_idx,
        'start_coords': start_coords,
        'start_z': start_z,
        'end_coords': end_coords,
        'end_z': end_z,
        'distance': distance,
        'elev_drop': elev_drop,
    })

    print(f"  ✓ Valid selection")
    print(f"    Name: {custom_name}")
    print(f"    Start: ({start_coords[0]:.2f}, {start_coords[1]:.2f}), Z={start_z:.1f}m")
    print(f"    End:   ({end_coords[0]:.2f}, {end_coords[1]:.2f}), Z={end_z:.1f}m")
    print(f"    Distance: {distance:.1f}m, Elev Drop: {elev_drop:.1f}m")
    print(f"    Neighbor Index: N{neighbor_idx}")

In [ ]:
# Save selected transects to CSV directly from transect_info
print("\n" + "=" * 100)
print("SAVING SELECTED TRANSECTS TO CSV")
print("=" * 100)

csv_rows = []
available_ids = [t['hillslope_id'] for t in transect_info]

for selection in selected_hillslopes:
    hillslope_id = selection['hillslope_id']
    transect_name = selection.get('name', f'H{hillslope_id}')

    matching_transects = [t for t in transect_info if t['hillslope_id'] == hillslope_id]
    if len(matching_transects) == 0:
        print(f"  ✗ Hillslope {hillslope_id} not found. Available: {available_ids}")
        continue

    transect = matching_transects[0]

    csv_rows.append({
        'transect_name': transect_name,
        'site_id': selected_site_id,
        'point_id': 1,
        'hillslope_id': transect['hillslope_id'],
        'neighbor_idx': transect['neighbor_idx'],
        'start_x': transect['start_coords'][0],
        'start_y': transect['start_coords'][1],
        'start_z': transect['start_z'],
        'end_x': transect['end_coords'][0],
        'end_y': transect['end_coords'][1],
        'end_z': transect['end_z'],
        'distance_m': transect['distance'],
        'elev_drop_m': transect['elev_drop'],
    })

if len(csv_rows) > 0:
    summary_df = pd.DataFrame(csv_rows)
    csv_filename = f'../data-processed/{site_name}/startendcoords_{site_name}_summary_forobsite.csv'
    summary_df.to_csv(csv_filename, index=False)

    print(f"✓ Saved summary CSV: {csv_filename}")
    print("\nSaved rows:")
    print(summary_df)
else:
    print("No valid selected hillslopes were found, so no CSV was written.")

In [ ]:
# # Save to .mat file
# if len(transects_to_save) > 0:
#     from scipy.io import savemat
    
#     # Prepare data for MATLAB format
#     # Create arrays for start_coords and end_coords
#     start_coords_array = np.array([t['start_coords'] for t in transects_to_save])
#     end_coords_array = np.array([t['end_coords'] for t in transects_to_save])
    
#     # Create metadata dictionary
#     metadata = {
#         'names': [t['name'] for t in transects_to_save],
#         'point_ids': [t['point_id'] for t in transects_to_save],
#         'hillslope_ids': [t['hillslope_id'] for t in transects_to_save],
#         'neighbor_indices': [t['neighbor_idx'] for t in transects_to_save],
#         'distances': [t['distance'] for t in transects_to_save],
#         'elev_drops': [t['elev_drop'] for t in transects_to_save],
#         'start_elevations': [t['start_z'] for t in transects_to_save],
#         'end_elevations': [t['end_z'] for t in transects_to_save]
#     }
    
#     data_to_save = {
#         'start_coords': start_coords_array,
#         'end_coords': end_coords_array,
#         'metadata': metadata,
#         'num_transects': len(transects_to_save),
#         'selected_category': selected_category
#     }
    
#     # Save to MAT file
#     m2_mat_filename = f'../data-processed/{site_name}/startendcoords_{site_name}_multi.mat'
#     savemat(m2_mat_filename, data_to_save)
    
#     print("\n" + "="*100)
#     print("SAVE COMPLETE")
#     print("="*100)
#     print(f"✓ Saved {len(transects_to_save)} transect(s) to: {m2_mat_filename}")
#     print(f"\nFile contents:")
#     print(f"  - start_coords: [{len(transects_to_save)} x 2] array")
#     print(f"  - end_coords: [{len(transects_to_save)} x 2] array")
#     print(f"  - metadata: Dictionary with transect details")
#     print(f"  - num_transects: {len(transects_to_save)}")
#     print(f"  - selected_category: '{selected_category}'")
    
#     print(f"\nSaved transects:")
#     for i, t in enumerate(transects_to_save):
#         print(f"  [{i+1}] {t['name']}: Point {t['point_id']}, Hillslope {t['hillslope_id']}, "
#               f"Distance={t['distance']:.1f}m")
    
#     # Also save a human-readable summary CSV
#     csv_filename = f'../data-processed/{site_name}/startendcoords_{site_name}_multi_summary.csv'
#     summary_df = pd.DataFrame({
#         'transect_name': [t['name'] for t in transects_to_save],
#         'point_id': [t['point_id'] for t in transects_to_save],
#         'hillslope_id': [t['hillslope_id'] for t in transects_to_save],
#         'neighbor_idx': [t['neighbor_idx'] for t in transects_to_save],
#         'start_x': [t['start_coords'][0] for t in transects_to_save],
#         'start_y': [t['start_coords'][1] for t in transects_to_save],
#         'start_z': [t['start_z'] for t in transects_to_save],
#         'end_x': [t['end_coords'][0] for t in transects_to_save],
#         'end_y': [t['end_coords'][1] for t in transects_to_save],
#         'end_z': [t['end_z'] for t in transects_to_save],
#         'distance_m': [t['distance'] for t in transects_to_save],
#         'elev_drop_m': [t['elev_drop'] for t in transects_to_save]
#     })
#     summary_df.to_csv(csv_filename, index=False)
#     print(f"\n✓ Also saved summary CSV: {csv_filename}")
    
# else:
#     print("\n⚠️  WARNING: No valid transects to save!")
#     print("   Please check your point_id and hillslope_id selections.")

In [ ]:
# Visualize selected transects
if len(transects_to_save) > 0 and generate_plots:
    # Manual configuration for plot bounds
    dx = 2000/2
    dy = 1600/2
    
    # Use first transect as reference point
    ref_x = transects_to_save[0]['end_coords'][0]
    ref_y = transects_to_save[0]['end_coords'][1]
    
    xmin = ref_x - dx/2*1
    xmax = ref_x + dx/2*2
    ymin = ref_y - dy/2*2
    ymax = ref_y + dy/2*1
    
    fig, ax = plt.subplots(1,1, figsize=(10,8))
    
    # 2D mesh plot
    transform = dem_profile2['transform']
    rows, cols = dem2.shape
    left = transform.c
    right = transform.c + cols * transform.a
    top = transform.f
    bottom = transform.f + rows * transform.e
    extent = [left, right, bottom, top]
    
    im1 = ax.imshow(dem2, cmap='terrain', extent=extent, origin='upper')
    ax.set_xlabel('Easting (m)')
    ax.set_ylabel('Northing (m)')
    fig.colorbar(im1, ax=ax, orientation='horizontal', pad=0.1, label='Elevation (m)')
    
    watershed_workflow.plot.hucs(watershed, crs, ax=ax, color='k', linewidth=1)
    watershed_workflow.plot.rivers(rivers, crs, ax=ax, color='red', linewidth=1)
    
    # Define colors for multiple transects
    colors_list = ['orange', 'purple', 'cyan', 'yellow', 'lime', 'deeppink']
    
    # Plot each selected transect
    for i, transect in enumerate(transects_to_save):
        start_coords = transect['start_coords']
        end_coords = transect['end_coords']
        name = transect['name']
        distance = transect['distance']
        
        color = colors_list[i % len(colors_list)]
        
        # Plot transect line
        ax.plot([start_coords[0], end_coords[0]], [start_coords[1], end_coords[1]], 
                color=color, linewidth=3, label=f'{name} ({distance:.0f}m)', alpha=0.8)
        
        # Plot start point
        ax.plot(start_coords[0], start_coords[1], '^', color=color, 
                markersize=12, markeredgecolor='black', markeredgewidth=2, zorder=6)
        
        # Add text label at start point
        ax.text(start_coords[0], start_coords[1] + 50, name, 
                fontsize=9, color='black', ha='center', va='bottom',
                bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.8),
                zorder=7)
    
    # Plot end points (pour points) - they may be the same for multiple transects
    unique_ends = {}
    for transect in transects_to_save:
        end_key = (round(transect['end_coords'][0], 2), round(transect['end_coords'][1], 2))
        unique_ends[end_key] = transect['end_coords']
    
    for end_coords in unique_ends.values():
        end_xy = shapely.geometry.Point(end_coords[0], end_coords[1])
        watershed_workflow.plot.shplys(end_xy, crs, ax=ax, color='blue', marker='o', 
                                       markersize=100, label='Pour Point' if len(unique_ends) == 1 else '', zorder=5)
    
    ax.set_xlim(xmin, xmax)
    ax.set_ylim(ymin, ymax)
    
    # Legend
    handles, labels = ax.get_legend_handles_labels()
    # Remove duplicate "Pour Point" labels
    by_label = dict(zip(labels, handles))
    if '' in by_label:
        by_label['Pour Point'] = by_label.pop('')
    ax.legend(by_label.values(), by_label.keys(), loc='upper left', fontsize=9, framealpha=0.9)
    
    plt.title(f'Selected Hillslope Transects ({len(transects_to_save)} total)\nCategory: {selected_category}')
    plt.tight_layout()
    
else:
    if len(transects_to_save) == 0:
        print("No transects to visualize")
    elif not generate_plots:
        print("Plotting disabled (generate_plots = False)")

In [ ]:
# # ==============================================================================
# # USER INPUT: Select ONE hillslope to save as single transect
# # ==============================================================================
# # 
# # Select the index (0-based) of the hillslope from transects_to_save
# # For example: 
# #   0 = first hillslope
# #   1 = second hillslope
# #   etc.

# selected_single_index = 0  # MODIFY THIS to select which hillslope to save

# # ==============================================================================

# if len(transects_to_save) > 0:
#     from scipy.io import savemat
    
#     # Validate index
#     if selected_single_index < 0 or selected_single_index >= len(transects_to_save):
#         print(f"\n✗ ERROR: Invalid index {selected_single_index}")
#         print(f"   Valid range: 0 to {len(transects_to_save)-1}")
#         print(f"   Available transects:")
#         for i, t in enumerate(transects_to_save):
#             print(f"     [{i}] {t['name']}")
#     else:
#         # Get selected transect
#         selected_transect = transects_to_save[selected_single_index]
        
#         print("\n" + "="*100)
#         print("SAVING SINGLE HILLSLOPE TRANSECT")
#         print("="*100)
#         print(f"\nSelected: [{selected_single_index}] {selected_transect['name']}")
#         print(f"  Point ID: {selected_transect['point_id']}")
#         print(f"  Hillslope ID: {selected_transect['hillslope_id']}")
#         print(f"  Start: ({selected_transect['start_coords'][0]:.2f}, {selected_transect['start_coords'][1]:.2f}), Z={selected_transect['start_z']:.1f}m")
#         print(f"  End:   ({selected_transect['end_coords'][0]:.2f}, {selected_transect['end_coords'][1]:.2f}), Z={selected_transect['end_z']:.1f}m")
#         print(f"  Distance: {selected_transect['distance']:.1f}m")
#         print(f"  Elev Drop: {selected_transect['elev_drop']:.1f}m")
        
#         # Prepare data for single transect (compatible with original format)
#         single_transect_data = {
#             'start_coords': selected_transect['start_coords'],
#             'end_coords': selected_transect['end_coords'],
#             'metadata': {
#                 'name': selected_transect['name'],
#                 'point_id': selected_transect['point_id'],
#                 'hillslope_id': selected_transect['hillslope_id'],
#                 'neighbor_idx': selected_transect['neighbor_idx'],
#                 'distance': selected_transect['distance'],
#                 'elev_drop': selected_transect['elev_drop'],
#                 'start_elevation': selected_transect['start_z'],
#                 'end_elevation': selected_transect['end_z']
#             },
#             'selected_category': selected_category
#         }
        
#         # Save to MAT file
#         single_mat_filename = f'../data-processed/{site_name}/startendcoords_{site_name}.mat'
#         savemat(single_mat_filename, single_transect_data)
        
#         print(f"\n✓ Saved single transect to: {single_mat_filename}")
#         print(f"\nFile contents:")
#         print(f"  - start_coords: [2] array")
#         print(f"  - end_coords: [2] array")
#         print(f"  - metadata: Dictionary with transect details")
#         print(f"  - selected_category: '{selected_category}'")
        
#         # Also save a human-readable CSV
#         csv_filename = f'../data-processed/{site_name}/startendcoords_{site_name}_summary.csv'
#         summary_df = pd.DataFrame({
#             'transect_name': [selected_transect['name']],
#             'point_id': [selected_transect['point_id']],
#             'hillslope_id': [selected_transect['hillslope_id']],
#             'neighbor_idx': [selected_transect['neighbor_idx']],
#             'start_x': [selected_transect['start_coords'][0]],
#             'start_y': [selected_transect['start_coords'][1]],
#             'start_z': [selected_transect['start_z']],
#             'end_x': [selected_transect['end_coords'][0]],
#             'end_y': [selected_transect['end_coords'][1]],
#             'end_z': [selected_transect['end_z']],
#             'distance_m': [selected_transect['distance']],
#             'elev_drop_m': [selected_transect['elev_drop']]
#         })
#         summary_df.to_csv(csv_filename, index=False)
#         print(f"✓ Also saved summary CSV: {csv_filename}")
        
# else:
#     print("\n⚠️  WARNING: No transects available to save!")
#     print("   Please run the previous cell to select and save multiple transects first.")